In [1]:
import sys
print(sys.executable)

C:\Python314\python.exe


In [2]:
# =============================================================================
# Hyperparameter Search: KSC (Kennedy Space Center)
#
# Searches LR_QUANTUM over a focused range with 2 seeds per config.
# Selects winner by mean validation accuracy. Test set is NEVER touched.
#
# Outputs the winning config so it can be pasted into
# AdaptiveReupload_KSC.py for final multi-seed test evaluation.
#
# Dependencies:
#   pip install numpy torch pennylane scipy scikit-learn
# =============================================================================

import time
GLOBAL_START = time.time()


def elapsed():
    return time.time() - GLOBAL_START


print("[Setup] Importing libraries...")
t0 = time.time()

import math
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pennylane as qml
from scipy.io import loadmat
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cpu")

print(f"[Setup] Device: {DEVICE}")
print(f"[Setup] PyTorch: {torch.__version__}, PennyLane: {qml.__version__}")
print(f"[Setup] Completed in {time.time()-t0:.2f}s | Total: {elapsed():.2f}s")


[Setup] Importing libraries...
[Setup] Device: cpu
[Setup] PyTorch: 2.11.0+cpu, PennyLane: 0.44.1
[Setup] Completed in 6.08s | Total: 6.08s


In [3]:
# =============================================================================
# HP SEARCH CONFIGURATION
# =============================================================================
HP_CONFIGS = [
    {"name": "lr_q_5e-4", "LR_QUANTUM": 5e-4},
    {"name": "lr_q_1e-3", "LR_QUANTUM": 1e-3},   # current default
    {"name": "lr_q_5e-3", "LR_QUANTUM": 5e-3},
    {"name": "lr_q_1e-2", "LR_QUANTUM": 1e-2},
]
SEARCH_SEEDS = [42, 0]

print(f"\n[HP Search] Configs: {len(HP_CONFIGS)}")
print(f"[HP Search] Seeds per config: {len(SEARCH_SEEDS)}")
print(f"[HP Search] Total runs: {len(HP_CONFIGS) * len(SEARCH_SEEDS)}")


[HP Search] Configs: 4
[HP Search] Seeds per config: 2
[HP Search] Total runs: 8


In [4]:
# =============================================================================
# DATASET LOADING
# =============================================================================
print("\n" + "=" * 60)
print("[Data] Loading KSC (Kennedy Space Center) dataset...")
t0 = time.time()

# --- UPDATE THESE PATHS for your machine ---
DATA_PATH = r"kennedy space center\KSC_corrected.mat"
GT_PATH = r"kennedy space center\KSC_gt.mat"

data_mat = loadmat(DATA_PATH)
gt_mat = loadmat(GT_PATH)
data_keys = [k for k in data_mat.keys() if not k.startswith("__")]
gt_keys = [k for k in gt_mat.keys() if not k.startswith("__")]
data_raw = data_mat[data_keys[0]].astype(np.float32)
gt = gt_mat[gt_keys[0]].astype(np.int64)

H, W, N_BANDS = data_raw.shape
unique_labels = np.sort(np.unique(gt))
unique_labels = unique_labels[unique_labels > 0]
NUM_CLASSES = len(unique_labels)
LABEL_MAP = {orig: idx for idx, orig in enumerate(unique_labels)}

print(f"[Data] Image: ({H}, {W}, {N_BANDS}), Classes: {NUM_CLASSES}")
print(f"[Data] Labeled pixels: {np.count_nonzero(gt)}")


[Data] Loading KSC (Kennedy Space Center) dataset...
[Data] Image: (512, 614, 176), Classes: 13
[Data] Labeled pixels: 5211


In [5]:
# =============================================================================
# FIXED HYPERPARAMETERS (not searched)
# =============================================================================
N_QUBITS = 4
N_REUP_LAYERS = 4
N_SUB_ROTATIONS = 3
ROUTER_BOTTLENECK = 32
ENCODING_DIM = N_QUBITS * N_SUB_ROTATIONS * 3
TOTAL_THETAS = N_REUP_LAYERS * ENCODING_DIM
TOTAL_ENT_ANGLES = (N_REUP_LAYERS - 1) * N_QUBITS

SPATIAL_DIM = 32
FUSION_DIM = 64
DROPOUT = 0.2
PATCH_SIZE = 9
TRAIN_SAMPLES_PER_CLASS = 30
VAL_SAMPLES_PER_CLASS = 15
HALF = PATCH_SIZE // 2

BATCH_SIZE = 16
N_EPOCHS = 50
LR_CLASSICAL = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 5.0
TOPOLOGY_BETA = 0.05
FOCAL_GAMMA = 2.0

In [6]:



# =============================================================================
# PREPROCESSING
# =============================================================================
print("\n" + "=" * 60)
print("[Preprocess] Normalizing bands and extracting patches...")
t0 = time.time()

data_norm = np.zeros_like(data_raw)
for b in range(N_BANDS):
    band = data_raw[:, :, b]
    bmin, bmax = band.min(), band.max()
    if bmax - bmin > 0:
        data_norm[:, :, b] = (band - bmin) / (bmax - bmin)

data_padded = np.pad(data_norm, ((HALF, HALF), (HALF, HALF), (0, 0)), mode="reflect")

labeled_indices = np.argwhere(gt > 0)
N_labeled = len(labeled_indices)
patches = np.zeros((N_labeled, PATCH_SIZE, PATCH_SIZE, N_BANDS), dtype=np.float32)
spectra = np.zeros((N_labeled, N_BANDS), dtype=np.float32)
labels = np.zeros(N_labeled, dtype=np.int64)

for idx, (i, j) in enumerate(labeled_indices):
    patches[idx] = data_padded[i:i + PATCH_SIZE, j:j + PATCH_SIZE, :]
    spectra[idx] = data_norm[i, j, :]
    labels[idx] = LABEL_MAP[gt[i, j]]

print(f"[Preprocess] Extracted {N_labeled} patches in {time.time()-t0:.2f}s")


# =============================================================================
# QUANTUM CIRCUIT
# =============================================================================
dev = qml.device("default.qubit", wires=N_QUBITS)


@qml.qnode(dev, interface="torch", diff_method="backprop")
def adaptive_reupload_circuit(enc_angles, thetas, ent_angles):
    for l in range(N_REUP_LAYERS):
        for q in range(N_QUBITS):
            for s in range(N_SUB_ROTATIONS):
                idx = (l * N_QUBITS * N_SUB_ROTATIONS + q * N_SUB_ROTATIONS + s) * 3
                qml.Rot(
                    thetas[idx]     + enc_angles[idx],
                    thetas[idx + 1] + enc_angles[idx + 1],
                    thetas[idx + 2] + enc_angles[idx + 2],
                    wires=q
                )
        if l < N_REUP_LAYERS - 1:
            for q in range(N_QUBITS):
                ent_idx = l * N_QUBITS + q
                qml.CRZ(ent_angles[ent_idx], wires=[q, (q + 1) % N_QUBITS])
    return [qml.expval(qml.PauliZ(q)) for q in range(N_QUBITS)]


# =============================================================================
# MODEL DEFINITION
# =============================================================================
class AdaptiveQuantumBranch(nn.Module):
    def __init__(self, n_bands, n_qubits, n_reup_layers, n_sub_rotations,
                 bottleneck=32):
        super().__init__()
        self.n_bands = n_bands
        self.n_qubits = n_qubits
        self.n_reup_layers = n_reup_layers
        self.n_sub_rotations = n_sub_rotations
        encoding_dim = n_qubits * n_sub_rotations * 3

        self.band_routers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(n_bands, bottleneck),
                nn.ReLU(),
                nn.Linear(bottleneck, n_bands),
                nn.Sigmoid()
            ) for _ in range(n_reup_layers)
        ])
        self.projections = nn.ModuleList([
            nn.Linear(n_bands, encoding_dim) for _ in range(n_reup_layers)
        ])
        self.ent_routers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(n_bands, bottleneck),
                nn.ReLU(),
                nn.Linear(bottleneck, n_qubits),
                nn.Tanh()
            ) for _ in range(n_reup_layers - 1)
        ])
        total_thetas = n_reup_layers * encoding_dim
        self.thetas = nn.Parameter(torch.randn(total_thetas) * 0.1)

    def forward(self, x):
        batch_size = x.shape[0]
        routing_weights, ent_weights = [], []
        all_enc, all_ent = [], []

        for l in range(self.n_reup_layers):
            alpha = self.band_routers[l](x)
            routing_weights.append(alpha)
            z = alpha * x
            enc = self.projections[l](z) * math.pi
            all_enc.append(enc)
            if l < self.n_reup_layers - 1:
                phi = self.ent_routers[l](x) * math.pi
                ent_weights.append(phi)
                all_ent.append(phi)

        flat_enc = torch.cat(all_enc, dim=1)
        flat_ent = torch.cat(all_ent, dim=1)
        q_features = torch.zeros(batch_size, self.n_qubits, device=x.device)
        for i in range(batch_size):
            result = adaptive_reupload_circuit(
                flat_enc[i], self.thetas, flat_ent[i]
            )
            q_features[i] = torch.stack(result)
        return q_features, routing_weights, ent_weights


class SpatialBranch(nn.Module):
    def __init__(self, n_bands, spatial_dim=32):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=(7, 3, 3), stride=(3, 1, 1), padding=(3, 1, 1)),
            nn.BatchNorm3d(8),
            nn.ReLU(),
            nn.Conv3d(8, 16, kernel_size=(5, 3, 3), stride=(2, 1, 1), padding=(2, 1, 1)),
            nn.BatchNorm3d(16),
            nn.ReLU(),
            nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(2, 1, 1), padding=(1, 1, 1)),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )
        self.fc = nn.Linear(32, spatial_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.relu(self.fc(x))


class AdaptiveReuploadClassifier(nn.Module):
    def __init__(self, n_bands, n_qubits, n_reup_layers, n_sub_rotations,
                 n_classes, router_bottleneck=32, spatial_dim=32,
                 fusion_dim=64, dropout=0.2):
        super().__init__()
        self.quantum_branch = AdaptiveQuantumBranch(
            n_bands, n_qubits, n_reup_layers, n_sub_rotations,
            router_bottleneck
        )
        self.spatial_branch = SpatialBranch(n_bands, spatial_dim)
        self.q_bn = nn.BatchNorm1d(n_qubits)
        self.s_bn = nn.BatchNorm1d(spatial_dim)
        fusion_input = n_qubits + spatial_dim
        self.classifier = nn.Sequential(
            nn.Linear(fusion_input, fusion_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, n_classes)
        )

    def forward(self, patches, spectra):
        q_features, routing_weights, ent_weights = self.quantum_branch(spectra)
        s_features = self.spatial_branch(patches)
        q_normed = self.q_bn(q_features)
        s_normed = self.s_bn(s_features)
        fused = torch.cat([q_normed, s_normed], dim=1)
        logits = self.classifier(fused)
        return logits, routing_weights, ent_weights


# =============================================================================
# LOSSES
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none")
        p_t = torch.exp(-ce)
        return ((1.0 - p_t) ** self.gamma * ce).mean()


def topology_diversity_loss(ent_weights, labels, n_classes, beta=0.05):
    if not ent_weights:
        return torch.tensor(0.0)
    phi = torch.cat(ent_weights, dim=1)
    class_means = []
    for c in range(n_classes):
        mask = labels == c
        if mask.sum() > 0:
            class_means.append(phi[mask].mean(dim=0))
    if len(class_means) < 2:
        return torch.zeros(1, device=phi.device).squeeze()
    class_means_t = torch.stack(class_means)
    inter_class_var = class_means_t.var(dim=0).mean()
    total_var = phi.var(dim=0).mean().clamp(min=1e-8)
    eta_sq = (inter_class_var / total_var).clamp(max=1.0)
    return -beta * eta_sq


def set_eval_with_bn_train(module):
    module.eval()
    for m in module.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm3d)):
            m.training = True


# =============================================================================
# TRAIN-ONE-RUN FUNCTION
# =============================================================================
def train_one_run(seed, lr_quantum):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    rng = np.random.RandomState(seed)
    train_sel, val_sel = [], []
    for c in range(NUM_CLASSES):
        class_idx = np.where(labels == c)[0]
        rng.shuffle(class_idx)
        n_train = min(TRAIN_SAMPLES_PER_CLASS, max(1, len(class_idx) // 3))
        n_val = min(VAL_SAMPLES_PER_CLASS, max(1, (len(class_idx) - n_train) // 2))
        train_sel.extend(class_idx[:n_train].tolist())
        val_sel.extend(class_idx[n_train:n_train + n_val].tolist())
    train_sel = np.array(train_sel, dtype=np.int64)
    val_sel = np.array(val_sel, dtype=np.int64)
    rng.shuffle(train_sel)
    rng.shuffle(val_sel)

    train_patches_t = torch.FloatTensor(patches[train_sel].transpose(0, 3, 1, 2)[:, np.newaxis])
    val_patches_t   = torch.FloatTensor(patches[val_sel].transpose(0, 3, 1, 2)[:, np.newaxis])
    train_spectra_t = torch.FloatTensor(spectra[train_sel])
    val_spectra_t   = torch.FloatTensor(spectra[val_sel])
    train_labels_t  = torch.LongTensor(labels[train_sel])
    val_labels_t    = torch.LongTensor(labels[val_sel])

    train_ds = TensorDataset(train_patches_t, train_spectra_t, train_labels_t)
    val_ds   = TensorDataset(val_patches_t,   val_spectra_t,   val_labels_t)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

    model = AdaptiveReuploadClassifier(
        n_bands=N_BANDS, n_qubits=N_QUBITS,
        n_reup_layers=N_REUP_LAYERS, n_sub_rotations=N_SUB_ROTATIONS,
        n_classes=NUM_CLASSES, router_bottleneck=ROUTER_BOTTLENECK,
        spatial_dim=SPATIAL_DIM, fusion_dim=FUSION_DIM, dropout=DROPOUT
    ).to(DEVICE)

    quantum_params = list(model.quantum_branch.parameters())
    classical_params = (
        list(model.spatial_branch.parameters()) +
        list(model.q_bn.parameters()) +
        list(model.s_bn.parameters()) +
        list(model.classifier.parameters())
    )
    optimizer = optim.Adam([
        {"params": quantum_params,   "lr": lr_quantum,     "weight_decay": 0},
        {"params": classical_params, "lr": LR_CLASSICAL,   "weight_decay": WEIGHT_DECAY}
    ])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
    criterion = FocalLoss(gamma=FOCAL_GAMMA)

    best_val_acc = -1.0

    for epoch in range(1, N_EPOCHS + 1):
        ep_start = time.time()
        model.train()
        train_correct, train_total = 0, 0
        for patches_b, spectra_b, labels_b in train_loader:
            patches_b = patches_b.to(DEVICE)
            spectra_b = spectra_b.to(DEVICE)
            labels_b = labels_b.to(DEVICE)

            optimizer.zero_grad()
            logits, _, ent_w = model(patches_b, spectra_b)
            cls_loss = criterion(logits, labels_b)
            topo_loss = topology_diversity_loss(ent_w, labels_b, NUM_CLASSES, TOPOLOGY_BETA)
            loss = cls_loss + topo_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()

            train_correct += (logits.argmax(1) == labels_b).sum().item()
            train_total += labels_b.size(0)
        train_acc = train_correct / train_total

        set_eval_with_bn_train(model)
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for patches_b, spectra_b, labels_b in val_loader:
                logits, _, _ = model(
                    patches_b.to(DEVICE), spectra_b.to(DEVICE)
                )
                val_correct += (logits.argmax(1).cpu() == labels_b).sum().item()
                val_total += labels_b.size(0)
        val_acc = val_correct / val_total

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        scheduler.step()
        ep_time = time.time() - ep_start
        print(f"    [seed={seed} lr_q={lr_quantum:.0e}] Ep {epoch:02d}/{N_EPOCHS} | "
              f"train={train_acc*100:.1f}% val={val_acc*100:.1f}% | {ep_time:.1f}s",
              end="\r")
    print()

    return best_val_acc


# =============================================================================
# HP SEARCH LOOP
# =============================================================================
print("\n" + "=" * 60)
print("[HP Search] Starting search...")
print(f"[HP Search] {len(HP_CONFIGS)} configs x {len(SEARCH_SEEDS)} seeds x {N_EPOCHS} epochs")
search_start = time.time()

results = {cfg["name"]: {} for cfg in HP_CONFIGS}

for cfg in HP_CONFIGS:
    for seed in SEARCH_SEEDS:
        run_start = time.time()
        print(f"\n[HP Search] Config={cfg['name']} | Seed={seed}")
        val_acc = train_one_run(seed=seed, lr_quantum=cfg["LR_QUANTUM"])
        results[cfg["name"]][seed] = val_acc
        print(f"[HP Search] Config={cfg['name']} Seed={seed} -> "
              f"val_acc={val_acc*100:.2f}% | elapsed {(time.time()-run_start)/60:.1f}m")

search_time = time.time() - search_start


# =============================================================================
# AGGREGATE AND REPORT
# =============================================================================
print("\n" + "=" * 60)
print("[HP Search] RESULTS")
print("=" * 60)
print(f"\n{'Config':<15s} " +
      " ".join(f"seed={s:>3d}" for s in SEARCH_SEEDS) +
      f"  {'Mean':>7s}  {'Std':>6s}")
print("-" * 60)

config_means = []
for cfg in HP_CONFIGS:
    accs = [results[cfg["name"]][s] for s in SEARCH_SEEDS]
    mean_acc = np.mean(accs)
    std_acc = np.std(accs)
    config_means.append((cfg, mean_acc, std_acc, accs))
    acc_str = " ".join(f"{a*100:>7.2f}%" for a in accs)
    print(f"{cfg['name']:<15s} {acc_str}  {mean_acc*100:>6.2f}%  {std_acc*100:>5.2f}%")

winner = max(config_means, key=lambda x: x[1])
winner_cfg, winner_mean, winner_std, winner_accs = winner

print("\n" + "=" * 60)
print(f"[HP Search] WINNER: {winner_cfg['name']}")
print(f"  LR_QUANTUM  = {winner_cfg['LR_QUANTUM']}")
print(f"  Mean val OA = {winner_mean*100:.2f}% ± {winner_std*100:.2f}% "
      f"(over {len(SEARCH_SEEDS)} seeds)")
print("=" * 60)

sorted_means = sorted([m for _, m, _, _ in config_means], reverse=True)
gap_top2 = (sorted_means[0] - sorted_means[1]) * 100
print(f"\n[HP Search] Diagnostic:")
print(f"  Gap between top 2 configs: {gap_top2:.2f} percentage points")
if gap_top2 < 1.0:
    print(f"  -> Ranking is noise; configs are ~equivalent. Use default (1e-3).")
elif gap_top2 < 3.0:
    print(f"  -> Weak signal. Winner is plausibly better but verify with extra seed.")
else:
    print(f"  -> Clear winner. Use winning config with confidence.")

print(f"\n[HP Search] Paste into AdaptiveReupload_KSC.py:")
print(f"  LR_QUANTUM = {winner_cfg['LR_QUANTUM']}")

print(f"\n[HP Search] Total time: {search_time/60:.1f} min | Wall time: {elapsed()/60:.1f} min")


[Preprocess] Normalizing bands and extracting patches...
[Preprocess] Extracted 5211 patches in 0.77s

[HP Search] Starting search...
[HP Search] 4 configs x 2 seeds x 50 epochs

[HP Search] Config=lr_q_5e-4 | Seed=42
    [seed=42 lr_q=5e-04] Ep 01/50 | train=27.1% val=32.3% | 42.5s
    [seed=42 lr_q=5e-04] Ep 02/50 | train=35.2% val=39.0% | 42.1s
    [seed=42 lr_q=5e-04] Ep 03/50 | train=43.5% val=49.2% | 43.7s
    [seed=42 lr_q=5e-04] Ep 04/50 | train=49.5% val=55.4% | 45.6s
    [seed=42 lr_q=5e-04] Ep 05/50 | train=56.8% val=51.8% | 43.3s
    [seed=42 lr_q=5e-04] Ep 06/50 | train=64.1% val=62.1% | 43.6s
    [seed=42 lr_q=5e-04] Ep 07/50 | train=65.1% val=59.5% | 42.2s
    [seed=42 lr_q=5e-04] Ep 08/50 | train=66.9% val=64.1% | 42.7s
    [seed=42 lr_q=5e-04] Ep 09/50 | train=71.1% val=68.7% | 42.6s
    [seed=42 lr_q=5e-04] Ep 10/50 | train=74.5% val=70.3% | 42.7s
    [seed=42 lr_q=5e-04] Ep 11/50 | train=73.7% val=77.4% | 42.4s
    [seed=42 lr_q=5e-04] Ep 12/50 | train=77.6% val=73.